In [ ]:
import os, subprocess, shutil

REPO_URL = "https://github.com/latamclaude/autoresearch-t4.git"
REPO_DIR = "/content/autoresearch-t4"  # absolute path — safe for Cell 1 re-runs
DATA_DIR = os.path.expanduser("~/.cache/autoresearch/data")
TRAIN_SHARD = os.path.join(DATA_DIR, "shard_00000.parquet")
VAL_SHARD = os.path.join(DATA_DIR, "shard_06542.parquet")
TRAIN_FRAC = 0.95

# 1. GPU check
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)

# 2. Clone repo (idempotent — uses absolute path)
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

# 3. Install uv + sync deps
if not shutil.which("uv"):
    subprocess.run(["pip", "install", "uv", "--quiet"], check=True)
subprocess.run(["uv", "sync"], check=True)

# 4. Download TinyStories → parquet shards (if not cached)
if not os.path.exists(TRAIN_SHARD) or not os.path.exists(VAL_SHARD):
    subprocess.run(["pip", "install", "datasets", "--quiet"], check=True)
    from datasets import load_dataset
    os.makedirs(DATA_DIR, exist_ok=True)
    ds = load_dataset("roneneldan/TinyStories", split="train")
    split = ds.train_test_split(test_size=1 - TRAIN_FRAC, seed=42)
    split["train"].to_parquet(TRAIN_SHARD)
    split["test"].to_parquet(VAL_SHARD)
    del ds, split  # free memory before training
    print("Data shards written.")

# 5. Train tokenizer (prepare.py checks if already trained)
result = subprocess.run(["uv", "run", "prepare.py"],
                        capture_output=True, text=True, cwd=REPO_DIR)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr[-2000:])
    raise RuntimeError("prepare.py failed — see error above. Do not run Cell 2.")

print("Setup complete.")

In [ ]:
!cd /content/autoresearch-t4 && git pull --ff-only && uv run train.py